# 06 · Building Wheels (`.whl`) & Source Distributions (sdist)

**Module 08 – Modules, Packaging & Dependency Management** · Estimated time: 75 min

> ❗ Not in either source. This lesson shows what actually gets uploaded to PyPI and installed by pip/uv, and how to build it yourself.

### 🎯 What you will learn
- Distributions: **sdist** (`.tar.gz`) vs **wheel** (`.whl`)
- Build **frontends** (`python -m build`, `uv build`, `poetry build`, `pip wheel`) vs **backends** (setuptools, hatchling, uv_build, poetry-core...)
- Building a real package step by step: src layout, package data, `py.typed`, console scripts, version
- **Inside a wheel**: `METADATA`, `WHEEL`, `RECORD`, `entry_points.txt`, `.dist-info`
- **Wheel filenames and tags**: `py3-none-any` vs `cp315-cp315-macosx_..._arm64`, `abi3`, manylinux/musllinux
- Building a **compiled C-extension wheel**
- Installing wheels, **editable installs** (PEP 660), verifying builds
- Producing wheels for every platform (cibuildwheel), reproducibility tips

In [1]:
import os, sys, tempfile, pathlib, shutil, subprocess
LAB = pathlib.Path(tempfile.mkdtemp(prefix="build-lab-"))
os.environ.update({
    "LAB": str(LAB), "PY": sys.executable,
    "PATH": os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"],
    "UV_CACHE_DIR": str(LAB / ".uv-cache"), "UV_PYTHON": sys.executable,
    "UV_PYTHON_DOWNLOADS": "never", "UV_NO_PROGRESS": "1", "NO_COLOR": "1",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1",
})
for var in ("FORCE_COLOR", "CLICOLOR_FORCE"):
    os.environ.pop(var, None)
print(LAB)

/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/build-lab-yxezayfp


## 1. sdist vs wheel

| | **sdist** (source distribution) | **wheel** (built distribution) |
|---|---|---|
| File | `textkit-1.0.0.tar.gz` | `textkit-1.0.0-py3-none-any.whl` |
| Contains | source code + `pyproject.toml` + `PKG-INFO` | files laid out **ready to copy** into `site-packages` + `.dist-info` metadata |
| Installing it | must be **built** first (runs the backend, may need a compiler) | just unzip + copy: fast, no code runs |
| Platform | universal | universal (`any`) **or** specific (OS/CPU/Python ABI) |
| Purpose | the "canonical" source; lets anyone rebuild | what installers prefer |

✅ Publish **both**: a wheel for fast installs and an sdist as the fallback/source of truth.

## 2. Frontends vs backends (PEP 517)

```
 FRONTEND (you run it)                          BACKEND (declared in [build-system])
 python -m build   ┐                            ┌─ setuptools.build_meta
 uv build          ├── isolated env + ────────► ├─ hatchling.build
 poetry build      │   build-system.requires    ├─ uv_build
 pip wheel / pip install                        ├─ poetry.core.masonry.api
                   ┘                            └─ flit_core / pdm.backend / maturin / scikit-build-core
```
The frontend creates an isolated environment, installs `build-system.requires`, and calls the backend's `build_sdist()` / `build_wheel()` hooks.

## 3. Build a real package: `textkit`

In [2]:
%%bash
cd "$LAB"
mkdir -p textkit/src/textkit/data textkit/tests
cd textkit

cat > pyproject.toml <<'EOF'
[build-system]
requires = ["setuptools>=77"]
build-backend = "setuptools.build_meta"

[project]
name = "textkit"
dynamic = ["version"]                       # read from the package (below)
description = "Tiny text utilities"
readme = "README.md"
requires-python = ">=3.12"
license = "MIT"
license-files = ["LICENSE"]
authors = [{ name = "Asha Rao", email = "asha@example.com" }]
classifiers = ["Programming Language :: Python :: 3", "Typing :: Typed"]
dependencies = []

[project.optional-dependencies]
color = ["rich>=13"]

[project.scripts]
textkit = "textkit.cli:main"

[project.urls]
Homepage = "https://example.com/textkit"

[tool.setuptools.dynamic]
version = { attr = "textkit.__version__" }

[tool.setuptools.packages.find]
where = ["src"]

[tool.setuptools.package-data]
textkit = ["py.typed", "data/*.txt"]
EOF

cat > src/textkit/__init__.py <<'EOF'
"""Tiny text utilities."""
__version__ = "1.0.0"
from .core import word_count, stopwords
__all__ = ["word_count", "stopwords"]
EOF

cat > src/textkit/core.py <<'EOF'
from importlib.resources import files

def stopwords() -> set[str]:
    """Load the bundled stop-word list (package data)."""
    return set(files("textkit").joinpath("data/stopwords.txt").read_text(encoding="utf-8").split())

def word_count(text: str, *, skip_stopwords: bool = False) -> int:
    words = text.lower().split()
    if skip_stopwords:
        stop = stopwords()
        words = [w for w in words if w not in stop]
    return len(words)
EOF

cat > src/textkit/cli.py <<'EOF'
import sys
from . import word_count

def main() -> None:
    text = " ".join(sys.argv[1:]) or "the quick brown fox"
    print(f"{word_count(text)} words ({word_count(text, skip_stopwords=True)} without stop-words)")
EOF

printf "the\na\nan\nand\nof\n" > src/textkit/data/stopwords.txt
touch src/textkit/py.typed
printf "# textkit\n\nTiny text utilities.\n" > README.md
echo "MIT License (example)" > LICENSE
find . -type f | sort

./LICENSE
./README.md
./pyproject.toml
./src/textkit/__init__.py
./src/textkit/cli.py
./src/textkit/

core.py
./src/textkit/data/stopwords.txt
./src/textkit/py.typed


Key ideas used here:
- **`dynamic = ["version"]`** with `[tool.setuptools.dynamic]`: the version lives in one place (`__version__`)
- **Package data** (`data/stopwords.txt`, `py.typed`) must be declared, or it won't be included in the wheel
- Read package data with **`importlib.resources.files()`**, never with paths relative to `__file__` (that breaks inside zips)
- **`py.typed`** (PEP 561) tells type checkers your package ships type hints

## 4. Build with `python -m build`

`pip install build` (the reference frontend from PyPA):

In [3]:
%%bash
cd "$LAB/textkit"
"$PY" -m build 2>&1 | tail -3          # builds the sdist, then a wheel FROM that sdist (checks the sdist is complete!)
ls -l dist | awk '{print $5, $9}'

adding 'textkit-1.0.0.dist-info/RECORD'
removing build/bdist.macosx-11.0-arm64/wheel
Successfully bu

ilt textkit-1.0.0.tar.gz and textkit-1.0.0-py3-none-any.whl


 
2998 textkit-1.0.0-py3-none-any.whl
2293 textkit-1.0.0.tar.gz


Useful options: `--wheel` / `--sdist` (only one), `--outdir DIR`, `--no-isolation` (use the current env, faster but less safe), `--installer uv` (use uv to create the isolated env, much faster), `-C key=value` (backend config settings).

## 5. What's inside the sdist?

In [4]:
import tarfile
sdist = next((LAB / "textkit/dist").glob("*.tar.gz"))
with tarfile.open(sdist) as tf:
    for name in sorted(tf.getnames()):
        print(name)
    print("\n--- PKG-INFO (first lines) ---")
    print(tf.extractfile(f"{sdist.name[:-7]}/PKG-INFO").read().decode()[:400])

textkit-1.0.0
textkit-1.0.0/LICENSE
textkit-1.0.0/PKG-INFO
textkit-1.0.0/README.md
textkit-1.0.0/pyproject.toml
textkit-1.0.0/setup.cfg
textkit-1.0.0/src
textkit-1.0.0/src/textkit
textkit-1.0.0/src/textkit.egg-info
textkit-1.0.0/src/textkit.egg-info/PKG-INFO
textkit-1.0.0/src/textkit.egg-info/SOURCES.txt
textkit-1.0.0/src/textkit.egg-info/dependency_links.txt
textkit-1.0.0/src/textkit.egg-info/entry_points.txt
textkit-1.0.0/src/textkit.egg-info/requires.txt
textkit-1.0.0/src/textkit.egg-info/top_level.txt
textkit-1.0.0/src/textkit/__init__.py
textkit-1.0.0/src/textkit/cli.py
textkit-1.0.0/src/textkit/core.py
textkit-1.0.0/src/textkit/data
textkit-1.0.0/src/textkit/data/stopwords.txt
textkit-1.0.0/src/textkit/py.typed

--- PKG-INFO (first lines) ---
Metadata-Version: 2.4
Name: textkit
Version: 1.0.0
Summary: Tiny text utilities
Author-email: Asha Rao <asha@example.com>
License-Expression: MIT
Project-URL: Homepage, https://example.com/textkit
Classifier: Programming Language :: Python :

## 6. What's inside the wheel?

A wheel is just a **zip file** with a special name and layout:

In [5]:
import zipfile
wheel = next((LAB / "textkit/dist").glob("*.whl"))
with zipfile.ZipFile(wheel) as zf:
    for info in zf.infolist():
        print(f"{info.file_size:>6}  {info.filename}")
    dist_info = next(n.split("/")[0] for n in zf.namelist() if n.endswith(".dist-info/METADATA"))
    for meta_file in ["WHEEL", "entry_points.txt", "METADATA"]:
        print(f"\n===== {dist_info}/{meta_file} =====")
        print(zf.read(f"{dist_info}/{meta_file}").decode()[:600])

   127  textkit/__init__.py
   216  textkit/cli.py
   451  textkit/core.py
     0  textkit/py.typed
    16  textkit/data/stopwords.txt
    22  textkit-1.0.0.dist-info/licenses/LICENSE
   478  textkit-1.0.0.dist-info/METADATA
    91  textkit-1.0.0.dist-info/WHEEL
    45  textkit-1.0.0.dist-info/entry_points.txt
     8  textkit-1.0.0.dist-info/top_level.txt
   853  textkit-1.0.0.dist-info/RECORD

===== textkit-1.0.0.dist-info/WHEEL =====
Wheel-Version: 1.0
Generator: setuptools (84.2.0)
Root-Is-Purelib: true
Tag: py3-none-any



===== textkit-1.0.0.dist-info/entry_points.txt =====
[console_scripts]
textkit = textkit.cli:main


===== textkit-1.0.0.dist-info/METADATA =====
Metadata-Version: 2.4
Name: textkit
Version: 1.0.0
Summary: Tiny text utilities
Author-email: Asha Rao <asha@example.com>
License-Expression: MIT
Project-URL: Homepage, https://example.com/textkit
Classifier: Programming Language :: Python :: 3
Classifier: Typing :: Typed
Requires-Python: >=3.12
Description-Content-Type:

| File in `*.dist-info/` | Purpose |
|---|---|
| `METADATA` | the project metadata (name, version, dependencies as `Requires-Dist`, extras as `Provides-Extra`, README as the long description). This is what PyPI displays |
| `WHEEL` | wheel format version, the generator, `Root-Is-Purelib`, the tags |
| `RECORD` | every file with its **sha256 hash and size**, used to verify and uninstall |
| `entry_points.txt` | console scripts and plugins |
| `licenses/` | license files (PEP 639) |
| `top_level.txt` | importable top-level names (setuptools) |

In [6]:
with zipfile.ZipFile(wheel) as zf:
    print(zf.read(f"{dist_info}/RECORD").decode())

textkit/__init__.py,sha256=LWZ2CuK02GmmWFcCI9g89yObznhBWMFLi_RhNMqfaRI,127
textkit/cli.py,sha256=ffVsOHtN6AoeD3aVaoOItobx0wf7qYKLog7qnDtn-X8,216
textkit/core.py,sha256=QElCHIyaaojfutoO_QeTFnbOtBPn_Bw4zcvS0UNCwjQ,451
textkit/py.typed,sha256=47DEQpj8HBSa-_TImW-5JCeuQeRkm5NMpJWZG3hSuFU,0
textkit/data/stopwords.txt,sha256=r3I_hvnB35tNbfKPpwhg4Lw3TvKsaBljkVvu_bfjWB4,16
textkit-1.0.0.dist-info/licenses/LICENSE,sha256=IyGbkOD0DR3mcEgOHytoMX75KIRRLDfbRdsrJe3BPqk,22
textkit-1.0.0.dist-info/METADATA,sha256=C7ASwAoDfIuJ0dscirq9xZlaMFV1Tbq84QK-KEGq_nQ,478
textkit-1.0.0.dist-info/WHEEL,sha256=a-udL2m4V_6Yn8b-uKzWb12RjRxwwMo6ohYFM24tpaY,91
textkit-1.0.0.dist-info/entry_points.txt,sha256=Wiva_sK59NlZu2TlLT_yGWCkE3MhXlD9TNPo7wFRz-M,45
textkit-1.0.0.dist-info/top_level.txt,sha256=lpmgNU60irV4dqWOlDgsEGKgBLZ8TOTDJZz76Hq-NIg,8
textkit-1.0.0.dist-info/RECORD,,



## 7. Wheel filenames & platform tags

```
{name}-{version}(-{build})?-{python tag}-{abi tag}-{platform tag}.whl
 textkit-1.0.0-py3-none-any.whl                         ← pure Python: any Python 3, any OS
 numpy-2.5.4-cp315-cp315-macosx_14_0_arm64.whl          ← CPython 3.15 ABI, macOS 14+, Apple Silicon
 numpy-2.5.4-cp315-cp315-manylinux_2_28_x86_64.whl      ← Linux with glibc ≥ 2.28, Intel/AMD 64-bit
 cryptography-46.0.0-cp311-abi3-win_amd64.whl           ← stable ABI: one wheel for CPython 3.11+
 pkg-1.0-cp315-cp315t-...whl                            ← "t" = free-threaded CPython build
```

In [7]:
from packaging.utils import parse_wheel_filename
from packaging import tags

for name in [wheel.name,
             "numpy-2.5.4-cp315-cp315-macosx_14_0_arm64.whl",
             "cryptography-46.0.0-cp311-abi3-manylinux_2_34_x86_64.whl"]:
    dist, version, build, wheel_tags = parse_wheel_filename(name)
    print(f"{dist:<13} {str(version):<7} {sorted(str(t) for t in wheel_tags)}")

supported = list(tags.sys_tags())
print(f"\nThis interpreter accepts {len(supported)} tags, most preferred first:")
print([str(t) for t in supported[:5]], "...", [str(t) for t in supported[-2:]])

textkit       1.0.0   ['py3-none-any']
numpy         2.5.4   ['cp315-cp315-macosx_14_0_arm64']
cryptography  46.0.0  ['cp311-abi3-manylinux_2_34_x86_64']

This interpreter accepts 1503 tags, most preferred first:
['cp315-cp315-macosx_26_0_arm64', 'cp315-cp315-macosx_26_0_universal2', 'cp315-cp315-macosx_25_0_arm64', 'cp315-cp315-macosx_25_0_universal2', 'cp315-cp315-macosx_24_0_arm64'] ... ['py31-none-any', 'py30-none-any']


Installers pick the **most specific compatible** wheel; if none matches they fall back to the sdist and build it locally (which may need compilers).

| Tag part | Examples |
|---|---|
| Python | `py3` (any Python 3), `cp315` (CPython 3.15), `pp311` (PyPy) |
| ABI | `none`, `cp315`, `cp315t` (free-threaded), `abi3` (stable ABI) |
| Platform | `any`, `win_amd64`, `macosx_11_0_arm64`, `manylinux_2_28_x86_64`, `musllinux_1_2_aarch64` (Alpine) |

## 8. Install the wheel and use it

In [8]:
%%bash
cd "$LAB"
"$PY" -m venv tryit
tryit/bin/python -m pip install -q textkit/dist/*.whl
tryit/bin/textkit "the cat and the hat"                     # the console script from [project.scripts]
tryit/bin/python -c "import textkit; print(textkit.__version__, sorted(textkit.stopwords()))"
tryit/bin/python -m pip show -f textkit | sed -n '/Files:/,$p' | head -12

5 words (2 without stop-words)


1.0.0 ['a', 'an', 'and', 'of', 'the']


Files:
  ../../../bin/textkit
  textkit-1.0.0.dist-info/INSTALLER
  textkit-1.0.0.dist-info/METADATA


  textkit-1.0.0.dist-info/RECORD
  textkit-1.0.0.dist-info/REQUESTED
  textkit-1.0.0.dist-info/WHEE

L
  textkit-1.0.0.dist-info/direct_url.json
  textkit-1.0.0.dist-info/entry_points.txt
  textkit-1.0

.0.dist-info/licenses/LICENSE
  textkit-1.0.0.dist-info/top_level.txt
  textkit/__init__.py


## 9. Editable installs (PEP 660)

During development you want code changes to apply **without reinstalling**:

In [9]:
%%bash
cd "$LAB"
"$PY" -m venv devenv
devenv/bin/python -m pip install -q -e ./textkit
devenv/bin/python -c "import textkit; print(textkit.word_count('a b c'))"
sed -i.bak 's/return len(words)/return len(words) * 100   # edited!/' textkit/src/textkit/core.py
devenv/bin/python -c "import textkit; print(textkit.word_count('a b c'), '← change visible without reinstalling')"
mv textkit/src/textkit/core.py.bak textkit/src/textkit/core.py
ls devenv/lib/python3*/site-packages | grep -i textkit

3


300 ← change visible without reinstalling


__editable__.textkit-1.0.0.pth
textkit-1.0.0.dist-info


The editable install puts a small `.pth` file / import hook into `site-packages` that points at your `src/` folder instead of copying files. `uv sync` and `poetry install` install your project editable by default.

## 10. Same project, different backends

Swap only `[build-system]`; the `[project]` table stays the same. Let's rebuild `textkit` with **hatchling** and **uv_build**, using **uv** as the frontend:

In [10]:
%%bash
cd "$LAB"
for backend in hatchling uv_build; do
  rm -rf "tk-$backend"; cp -R textkit "tk-$backend"; cd "tk-$backend"; rm -rf dist build src/*.egg-info
  "$PY" - "$backend" <<'EOF'
import re, sys, pathlib
backend = sys.argv[1]
p = pathlib.Path("pyproject.toml"); t = p.read_text()
build_system = {
    "hatchling": '[build-system]\nrequires = ["hatchling"]\nbuild-backend = "hatchling.build"\n',
    "uv_build":  '[build-system]\nrequires = ["uv_build>=0.13,<0.14"]\nbuild-backend = "uv_build"\n',
}[backend]
t = re.sub(r"\[build-system\].*?build-backend = \"setuptools.build_meta\"\n", build_system, t, flags=re.S)
t = t.replace('dynamic = ["version"]', 'version = "1.0.0"')            # static version (simplest across backends)
t = t[: t.index("[tool.setuptools.dynamic]")]                          # drop setuptools-only settings
p.write_text(t)
EOF
  uv build 2>&1 | tail -2 | sed "s|^|[$backend] |"
  "$PY" -c "import zipfile,glob; z=zipfile.ZipFile(glob.glob('dist/*.whl')[0]); print('   data file included:', any('stopwords.txt' in n for n in z.namelist()))"
  cd ..
done

[hatchling] Successfully built dist/textkit-1.0.0.tar.gz
[hatchling] Successfully built dist/textkit

-1.0.0-py3-none-any.whl


   data file included: True


[uv_build] Successfully built dist/textkit-1.0.0.tar.gz
[uv_build] Successfully built dist/textkit-1

.0.0-py3-none-any.whl


   data file included: True


hatchling and uv_build include every file inside the package directory automatically (data files too); setuptools needed `package-data`. Each backend has its own `[tool.<backend>]` include/exclude settings for fine control.

Other frontends:
```bash
uv build                  # sdist + wheel, uses uv for isolation (fast)
poetry build              # Poetry projects (poetry-core backend)
pip wheel . -w dist/      # wheel only (also builds wheels for dependencies)
hatch build               # Hatch projects
```

## 11. A compiled C-extension wheel (platform-specific)

In [11]:
%%bash
cd "$LAB" && mkdir -p fastmath/src && cd fastmath
cat > src/fastmath.c <<'EOF'
#define PY_SSIZE_T_CLEAN
#include <Python.h>

/* fastmath.add(a, b): add two integers in C */
static PyObject *add(PyObject *self, PyObject *args) {
    long a, b;
    if (!PyArg_ParseTuple(args, "ll", &a, &b)) return NULL;
    return PyLong_FromLong(a + b);
}
static PyMethodDef methods[] = {{"add", add, METH_VARARGS, "Add two integers."}, {NULL, NULL, 0, NULL}};
static struct PyModuleDef module = {PyModuleDef_HEAD_INIT, "fastmath", "A tiny C extension.", -1, methods};
PyMODINIT_FUNC PyInit_fastmath(void) { return PyModule_Create(&module); }
EOF
cat > pyproject.toml <<'EOF'
[build-system]
requires = ["setuptools>=77"]
build-backend = "setuptools.build_meta"
[project]
name = "fastmath"
version = "0.1.0"
EOF
cat > setup.py <<'EOF'
from setuptools import setup, Extension
setup(ext_modules=[Extension("fastmath", ["src/fastmath.c"])])   # setup.py is still used for C extensions
EOF
"$PY" -m build --wheel 2>&1 | tail -1
ls dist
"$PY" -m venv cenv && cenv/bin/python -m pip install -q dist/*.whl
cenv/bin/python -c "import fastmath; print('fastmath.add(40, 2) =', fastmath.add(40, 2)); print(fastmath.__file__)"

Successfully built fastmath-0.1.0-cp315-cp315-macosx_11_0_arm64.whl


fastmath-0.1.0-cp315-cp315-macosx_11_0_arm64.whl


fastmath.add(40, 2) = 42
/private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/build-lab-yxezayfp

/fastmath/cenv/lib/python3.15/site-packages/fastmath.cpython-315-darwin.so


The wheel is tagged for **this exact Python ABI and platform** (`cp315-cp315-macosx_..._arm64`), and contains a compiled `.so` (`.pyd` on Windows). To support everyone you must build one wheel per platform/Python:

- **cibuildwheel** builds wheels for Linux (manylinux/musllinux), macOS and Windows in CI:
  ```yaml
  - uses: pypa/cibuildwheel@v3
  ```
- **abi3 / Stable ABI** (`py_limited_api=True`): one wheel works for many CPython versions
- Modern alternatives to raw C: **Cython**, **pybind11**/**nanobind** (C++), **PyO3 + maturin** (Rust), **mypyc** (compile typed Python), **scikit-build-core** (CMake). See Module 10 · 09.

## 12. Checking your distributions before publishing

In [12]:
%%bash
cd "$LAB/textkit"
"$PY" -m twine check dist/*

Checking dist/textkit-1.0.0-py3-none-any.whl: 

PASSED


Checking dist/textkit-1.0.0.tar.gz: 

PASSED


Checklist:
- ✅ `twine check dist/*` passes (README renders, metadata valid)
- ✅ Install the **wheel** into a fresh venv and import it / run its commands (as we did)
- ✅ Run your tests against the installed package, not your source folder (the src layout helps)
- ✅ The sdist contains everything needed to rebuild (`python -m build` checks this by building the wheel *from* the sdist)
- ✅ Version bumped; no stray files (`.env`, `__pycache__`, large data) in the archives

### Reproducible builds
Set `SOURCE_DATE_EPOCH` (timestamps), pin `build-system.requires`, and build in CI from a clean checkout.

## 📝 Summary
- **sdist** = source archive; **wheel** = ready-to-install zip with `.dist-info` metadata. Publish both.
- Frontends (`python -m build`, `uv build`, `poetry build`) call the backend declared in `[build-system]`.
- Declare package data; read it with `importlib.resources`; ship `py.typed`.
- Wheel tags (`py3-none-any`, `cp315-cp315-manylinux...`, `abi3`) decide where a wheel can be installed.
- Use editable installs for development, `twine check` + a clean-venv install before publishing.

## 🏋️ Practice
1. Add a second data file `data/emoji.txt` to `textkit`, rebuild, and verify it's in the wheel.
2. Change the version to `1.1.0b1`; how does the wheel filename change?
3. Build `textkit` with `poetry-core` as the backend (hint: `requires = ["poetry-core>=2"]`, `build-backend = "poetry.core.masonry.api"`, plus `[tool.poetry] packages = [{include = "textkit", from = "src"}]`).
4. Use `pip wheel requests -w wheelhouse/` and inspect the tags of every downloaded wheel with `parse_wheel_filename`.